# The genie game: deriving Robbins-Monro

CAS BMAI HS26, weekend 1, Friday 10:00. Slug `hs26-bmai-w1-opt-genie-rm`. TAs Jinlei and Suei-Wen.

Lost in the desert, you rub a lamp. The genie will set you free, on one condition:

> Name me a positive number that my own test accepts.

Its test is `Q -> 1 + 1/Q`. You are free when the number you hand in and the number that comes back are the same number. You may test as often as you like. You may not ask for the answer, and there is no computer algebra in this notebook.

By the end you will have invented the Robbins-Monro algorithm, which is the loop the 11:00 session calls stochastic gradient descent.

**Solutions notebook.** Every blank is filled in. Hand the student version to participants.

In [ ]:
# @title Setup: the genie. Run this cell, then leave it alone.
# NOTEBOOK_VERSION = "v1.0"
# This is exercise/genie.py, carried inline so the notebook needs
# no clone and no network. Do not edit it here.

"""The genie's bargain, as a Python object the notebooks program against.

    The bargain: name a positive Q with Q = 1 + 1/Q.
    You may test as often as you like. You may not ask for the answer.

WHY THIS FILE EXISTS AT ALL. The AINIT darts cartridge hands its participants
an exercise written against a class called DartsInTheDark, and that class is
not in the arcade repository: its browser game returns nothing to Python and
shares no code with the exercise. So this one is written from scratch, once,
and both the notebook and the page's scene 7 stub program against it.

WHY THE RANDOM NUMBERS ARE NOT random.gauss. The browser page draws its
sandstorm from Mulberry32 plus Box-Muller (js/rng.js). If this file used the
standard library, a participant running the notebook would see a different
storm from the one the room watched. So both generators are ported here line
for line, and _assert_stream_matches_browser() checks the port against six
draws taken from the running JavaScript at seed 20260904. That assert runs at
import time: a broken port fails immediately rather than quietly producing
plausible numbers.

    python3 exercise/genie.py        runs the three sanity checks and prints them

Sign convention, pinned for the whole exercise and matching the lecture slide:

    loss = Q - target                your number minus what came back
    Q    = Q - alpha * loss          subtract part of the loss

The house-price exercise 40 minutes later writes the same rule the other way
round, l = y - y_hat with w = w + alpha * l. Both move the estimate toward
the target. See ../CLAUDE.md, "Notation, pinned".
"""

import math

SEED = 20260904          # the weekend date. Pinned in js/rng.js as well.


# --------------------------------------------------------------------------
# The two generators, ported from js/rng.js
# --------------------------------------------------------------------------

def _imul(a, b):
    """JavaScript Math.imul: multiply the low 32 bits and keep 32 bits.

    Multiplication modulo 2^32 depends only on the residues, so working with
    unsigned values here gives the same bit pattern JavaScript gets from its
    signed 32 bit result.
    """
    return ((a & 0xFFFFFFFF) * (b & 0xFFFFFFFF)) & 0xFFFFFFFF


def mulberry32(seed):
    """One multiply-xorshift round per draw. Returns floats in [0, 1)."""
    state = seed & 0xFFFFFFFF

    def rand():
        nonlocal state
        state = (state + 0x6D2B79F5) & 0xFFFFFFFF
        t = state
        t = _imul(t ^ (t >> 15), t | 1)
        t = (t ^ (t + _imul(t ^ (t >> 7), t | 61))) & 0xFFFFFFFF
        return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296

    return rand


def gauss(rand, sd=1.0):
    """Box-Muller. Two uniforms in, one normal deviate out.

    u is drawn as 1 - rand() so that it is never exactly zero and log(u)
    cannot blow up. That detail is inherited from the browser code and it
    has to stay, because it changes which numbers come out.
    """
    u = 1 - rand()
    v = rand()
    return math.sqrt(-2 * math.log(u)) * math.cos(2 * math.pi * v) * sd


# Six draws taken from the running JavaScript, js/rng.js, at seed 20260904.
# Reproduce them with:
#     node -e 'const fs=require("fs");const w={};
#              new Function("window",fs.readFileSync("js/rng.js","utf8"))(w);
#              const r=w.RNG.stream();for(let i=0;i<6;i++)console.log(r());'
_BROWSER_UNIFORMS = [
    0.28361130738630891,
    0.70026570325717330,
    0.26360480696894228,
    0.12938253255560994,
    0.45396000216715038,
    0.40642114332877100,
]
_BROWSER_GAUSS_015 = [
    -0.03766342680316412,
    0.08065835197867159,
    -0.13729813445080333,
    -0.21030241155996920,
    -0.01691869574573623,
    0.19891862125225321,
]


def _assert_stream_matches_browser():
    r = mulberry32(SEED)
    for i, want in enumerate(_BROWSER_UNIFORMS):
        got = r()
        assert abs(got - want) < 1e-15, (
            f"uniform {i}: python {got!r} against browser {want!r}. "
            "The Mulberry32 port has drifted from js/rng.js."
        )
    r = mulberry32(SEED)
    for i, want in enumerate(_BROWSER_GAUSS_015):
        got = gauss(r, 0.15)
        assert abs(got - want) < 1e-15, (
            f"gauss {i}: python {got!r} against browser {want!r}. "
            "The Box-Muller port has drifted from js/rng.js."
        )


_assert_stream_matches_browser()


# --------------------------------------------------------------------------
# The bargain
# --------------------------------------------------------------------------

class Genie:
    """The genie. Hand it a number with ask(); it hands one back.

    form "A" is the bargain as it is first written, Q = 1 + 1/Q, so the test
    is 1 + 1/Q. form "B" is the same bargain rearranged, Q = 1/(Q - 1), so
    the test is 1/(Q - 1). Same equation, same solutions, and the loop does
    not behave the same on the two. That is the whole point of scene 3.

    sigma is the sandstorm. With sigma = 0 the genie answers cleanly; above
    zero its answer arrives blurred by a Gaussian draw, so the same number
    handed over twice comes back differently.
    """

    def __init__(self, start=1.0, sigma=0.0, seed=SEED, form="A"):
        self.start = start
        self.sigma = sigma
        self.seed = seed
        self.form = form
        self.reset()

    def reset(self):
        self.guess = self.start
        self.t = 0
        self._rng = mulberry32(self.seed)

    def test(self, q):
        """What the bargain demands of q, with no storm. Not for the loop."""
        return 1.0 + 1.0 / q if self.form == "A" else 1.0 / (q - 1.0)

    def ask(self, q):
        """Test a number. This is the only thing the genie will do for you."""
        if q <= 0:
            raise ValueError("The genie wants a positive number.")
        if self.form == "B" and q == 1:
            raise ValueError("The genie cannot divide by zero. Try another number.")
        clean = self.test(q)
        self.t += 1
        if self.sigma:
            return clean + gauss(self._rng, self.sigma)
        return clean


def schedule(t, alpha_0=1.0, decay_rate=5.0):
    """This round's step size: large early, small later."""
    return alpha_0 / (1 + t / decay_rate)


def robbins_monro(genie, n_rounds=60, alpha_0=1.0, decay_rate=5.0):
    """The reference solution. The notebook blanks the three lines marked."""
    guesses, losses, learning_rates = [], [], []
    genie.reset()

    for t in range(n_rounds):
        alpha_t = schedule(t, alpha_0, decay_rate)
        learning_rates.append(alpha_t)

        current = genie.guess
        guesses.append(current)

        target = genie.ask(current)

        loss = current - target                       # blank 1
        losses.append(loss)

        step = alpha_t * loss                         # blank 2

        genie.guess = current - step                  # blank 3

    return guesses, losses, learning_rates


# --------------------------------------------------------------------------
# The two readouts of the instrument panel. Both computable from what the
# participant heard, and neither one uses the answer.
# --------------------------------------------------------------------------

def last_third(values):
    return values[(len(values) * 2) // 3:] if values else []


def spread(guesses):
    """Does the loop settle? Standard deviation over the last third."""
    tail = last_third(guesses)
    if len(tail) < 2:
        return float("nan")
    m = sum(tail) / len(tail)
    return math.sqrt(sum((v - m) ** 2 for v in tail) / len(tail))


def average_heard_loss(losses):
    """Is the genie satisfied? Mean of the losses over the last third."""
    tail = last_third(losses)
    return sum(tail) / len(tail) if tail else float("nan")


# --------------------------------------------------------------------------
# Sanity checks. The FS26 convention is that a cheap check precedes the
# expensive run, and that the check catches a wrong SIGN first of all: writing
# loss = target - current while keeping guess = current - step sends the first
# update to exactly Q = 0, which the genie cannot even test.
# --------------------------------------------------------------------------

def sanity_1_deck_table():
    """Storm off, whole step, six rounds from Q = 1: the lecture slide."""
    g = Genie(start=1.0, sigma=0.0)
    guesses, losses, _ = robbins_monro(g, n_rounds=6, alpha_0=1.0, decay_rate=1e18)
    want_q = [1, 2, 1.5, 5 / 3, 1.6, 1.625]
    want_l = [-1, 0.5, -1 / 6, 1 / 15, -0.025, 1 / 104]
    for i, (a, b) in enumerate(zip(guesses, want_q)):
        assert abs(a - b) < 1e-12, f"row {i + 1} guess {a} against {b}"
    for i, (a, b) in enumerate(zip(losses, want_l)):
        assert abs(a - b) < 1e-12, f"row {i + 1} loss {a} against {b}"
    return guesses, losses


def sanity_2_clean_convergence():
    """Storm off, the shrinking step, sixty rounds. This is scene 6's run."""
    g = Genie(start=1.0, sigma=0.0)
    robbins_monro(g, n_rounds=60, alpha_0=1.0, decay_rate=5.0)
    assert f"{g.guess:.10f}" == "1.6180339887", f"{g.guess!r}"
    return g.guess


def sanity_3_noisy_convergence():
    """Storm on, sigma 0.15, 400 rounds, at the pinned seed."""
    g = Genie(start=1.0, sigma=0.15)
    guesses, losses, _ = robbins_monro(g, n_rounds=400, alpha_0=1.0, decay_rate=5.0)
    s = spread(guesses)
    a = average_heard_loss(losses)
    # generous on purpose: these thresholds exist to catch a wrong sign, and
    # not to grade the seed
    assert s < 0.02, f"spread {s}"
    assert abs(a) < 0.05, f"average heard loss {a}"
    return s, a, g.guess

print("The genie is ready.")

---
## Task 1: Hear what the genie says

You may test any positive number. Testing is free. Nothing else is on offer: the genie will not tell you the answer, and there is no formula in front of you.

Hand it three numbers of your own choosing and look at the third column. The loss is your number minus the number that came back.

**What you need to do**

1. Fill in the one blank so that `loss` is your number minus the target.
2. Run the cell. Try three numbers you pick yourself before reading on.

In [ ]:
# ----- Task 1: hear what the genie says -----
genie = Genie(start=1.0, sigma=0.0)

for q in [1.0, 2.0, 1.5]:
    target = genie.ask(q)
    # COMPLETE THIS LINE: your number minus what came back
    loss = q - target
    verdict = "too big" if loss > 0 else "too small"
    print(f"Q = {q:8.4f}   target = {target:8.4f}   "
          f"loss = {loss:+8.4f}   {verdict}")

### What just happened?

Every row is three numbers you could have written on paper. The verdict is the sign of the loss and nothing more.

Discussion, with no code:

- What would it mean for a number to have a loss of exactly zero?
- The genie never said which way to move. Where did the verdict come from?
- You now have a number in front of you that you did not have before you asked. What is the cheapest thing to do with it?

---
## Task 2: The whole step

The cheapest thing to do with the target is to hand it straight back. That move has two descriptions and they are the same instruction:

```
Q <- target          move onto the number that came back
Q <- Q - loss        subtract the whole loss
```

They agree because the loss is Q minus the target. Write the second one, because it is the one with a dial on it.

**What you need to do**

1. Fill in the blank with the second description, in terms of `current` and `loss`.
2. Run it. The six rows are the ones on the lecture slide.

In [ ]:
# ----- Task 2: the whole step -----
genie = Genie(start=1.0, sigma=0.0)
current = genie.guess

for t in range(6):
    target = genie.ask(current)
    loss = current - target
    print(f"{t+1}   Q = {current:10.6f}   target = {target:10.6f}   "
          f"loss = {loss:+10.6f}")
    # COMPLETE THIS LINE: subtract the whole loss
    current = current - loss

### What just happened?

Expected, and this is the check that catches a wrong sign first:

```
guesses  1  2  1.5  1.66667  1.6  1.625
losses   -1  +0.5  -0.166667  +0.0666667  -0.025  +0.00961538
```

Discussion, with no code:

- The losses alternate in sign. What does the guess do as a result?
- Writing `loss = target - current` while keeping `current = current - loss` sends the first update to exactly zero, and the genie cannot test zero. Why is that a useful way to fail?
- Nothing here is named yet. What have you actually got?

---
## Task 3: The same bargain, written the other way

Multiply the bargain by Q and it says `Q^2 = Q + 1`, so `Q(Q - 1) = 1`, so `Q = 1/(Q - 1)`. One equation, two ways of writing it, the same solutions.

Nothing else changes. Same whole step, same genie, same rules. This cell is written for you: run it and watch.

**What you need to do**

1. Run the cell as it stands.
2. Read the last line it prints before it stops.

In [ ]:
# ----- Task 3: form B, and the whole step -----
genie = Genie(start=1.7, sigma=0.0, form="B")
current = genie.guess

for t in range(8):
    try:
        target = genie.ask(current)
    except ValueError as refusal:
        print(f"{t+1}   Q = {current:10.4f}   {refusal}")
        break
    loss = current - target
    print(f"{t+1}   Q = {current:10.4f}   target = {target:10.4f}   "
          f"loss = {loss:+10.4f}")
    current = current - loss

### What just happened?

The run goes

```
1.7000 -> 1.4286 -> 2.3333 -> 0.7500 -> -4.0000
```

and after four tests it has handed over -4.0, which the bargain refuses. The whole step did not fail because the bargain changed. It failed because the bargain was written differently.

Discussion, with no code:

- Task 2 and task 3 solve the same equation. Which one was lucky?
- The loop is running away from something. What?
- You have one dial you have not turned yet.

---
## Task 4: Move only part of the way

The whole step subtracts the whole loss. Subtract part of it instead, and call the fraction alpha:

```
Q <- Q - alpha * loss
```

With alpha = 1 this is task 2 again. Run the sweep and find where form B starts to behave.

**What you need to do**

1. Fill in the update, using `alpha`.
2. Run it. Read which values of alpha settle and which do not.

In [ ]:
# ----- Task 4: a partial step -----
def run(alpha, form, start, n=200):
    genie = Genie(start=start, sigma=0.0, form=form)
    current = genie.guess
    for t in range(n):
        if current <= 0 or (form == "B" and current == 1):
            return None                # left the bargain
        target = genie.ask(current)
        loss = current - target
        # COMPLETE THIS LINE: subtract only alpha of the loss
        current = current - alpha * loss
    return current

for alpha in [0.2, 0.3, 0.5, 0.6, 0.7, 1.0]:
    out = run(alpha, "B", 1.7)
    print(f"form B   alpha = {alpha:4.2f}   "
          + ("left the bargain" if out is None else f"rests at {out:.9f}"))

### What just happened?

At alpha = 0.30 the loop on form B rests at 1.6180339887, which is where task 2 arrived by the other route. Moving part of the way does more than tidy up: it changes which answer the loop reaches.

Discussion, with no code:

- There is a value of alpha above which form B stops settling. Between which two of the values you ran does it sit?
- Try the same sweep on form A. Where is its edge, and why is it in a different place?
- Nothing has been noisy yet. Every number the genie returned was exact.

---
## Task 5: The sandstorm

Now the wind rises. The genie's answer arrives blurred, so the same number handed over twice comes back differently.

Two readouts, and both are computable from what you heard. Neither one uses the answer, because you still do not have it.

- **spread**: the standard deviation of your guesses over the last third of the run. Did the loop settle?
- **average heard loss**: the mean of the losses over the last third. Is the genie satisfied?

**What you need to do**

1. Fill in the update, using this round's `alpha_t`.
2. Run it. Compare the three schedules on both readouts.

In [ ]:
# ----- Task 5: three schedules under the storm -----
def storm(alpha_of_t, n=400):
    genie = Genie(start=1.0, sigma=0.15)
    current, guesses, losses = genie.guess, [], []
    for t in range(n):
        alpha_t = alpha_of_t(t)
        guesses.append(current)
        target = genie.ask(current)
        loss = current - target
        losses.append(loss)
        # COMPLETE THIS LINE: the same update, with this round's step
        current = current - alpha_t * loss
    return spread(guesses), average_heard_loss(losses), current

for name, rule in [("alpha = 1",         lambda t: 1.0),
                   ("alpha = 0.2",       lambda t: 0.2),
                   ("alpha = 1/(t+1)",   lambda t: 1.0 / (t + 1))]:
    s, a, last = storm(rule)
    print(f"{name:18s} spread {s:9.3e}   avg heard loss {a:+9.3e}   "
          f"rests at {last:.6f}")

### What just happened?

The constant step settles into a band and the band never closes, however long you run. Only the shrinking step closes it.

Discussion, with no code:

- Both readouts have to be small at once. What does a small spread with a large average loss describe?
- The storm never gets weaker. What is it about a shrinking step that makes the wobble cancel?
- Run the same three schedules a second time. Why is the answer identical?

---
## Task 6: Stretch: why not shrink faster

If shrinking is good, shrink hard. Add `alpha = 1/(t+1)^2` and look at both readouts before you decide it wins.

This cell is written for you.

**What you need to do**

1. Run it.
2. Compare its spread with the shrinking schedule from task 5.

In [ ]:
# ----- Task 6: the schedule that stops too early -----
s, a, last = storm(lambda t: 1.0 / (t + 1) ** 2)
print(f"alpha = 1/(t+1)^2  spread {s:9.3e}   avg heard loss {a:+9.3e}")
print(f"it rests at {last:.6f}, and the genie returns "
      f"{Genie(form='A').test(last):.6f} for that number")

### What just happened?

It has the smallest spread of all four, 6.973e-05, and an average heard loss of 1.867e-01. It rests at 1.749483, where the genie returns 1.571597. It settled beautifully, on a number the genie refuses, and it looks more settled than the row that is right.

So the step sizes have to satisfy two demands at once, and this is Robbins-Monro:

- they must add up without bound, so the loop can still travel from any start;
- their squares must add up to something finite, so the wobble eventually cancels.

`alpha_t = 1/t` satisfies both. `alpha_t = 1/t^2` fails the first, and the run above is what that failure looks like from the outside.

Discussion, with no code:

- Which of the two demands does a constant alpha fail?
- The clean statement is that when the steps add up to something finite, a fixed fraction of the starting error survives forever. Check it: run the fast schedule from a start far away, with the storm off, and see how much of the distance it covers.
- With the storm off and a shrinking step, the loop reaches 1.6180339887. That number has a name. What is it?

---
## Where this goes next

| the genie's loop | the model's loop |
|---|---|
| `Q`, your guess | `w`, the weight in the model |
| the genie's test, `1 + 1/Q` | what one house's price demands of `w` |
| `loss = Q - target` | the residual on that one house |
| `Q <- Q - alpha_t * loss` | the same update |
| the sandstorm | one house at a time instead of all of them |
| `alpha_t` shrinking | the learning rate schedule |
| solve it in closed form | there is no formula to solve |

The last row is the one that matters. For the genie the quadratic formula genuinely would have worked. For a model it does not exist, and that is why the loop is the method rather than a curiosity.

The browser page for this exercise is `../index.html`, opened straight from disk. Press `t` for dark mode and the arrow keys to step.